# Часть 1. EDA и тематическое моделирование
**Датасет:** `MonoHime/ru_sentiment_dataset` — русскоязычные отзывы с метками тональности.

In [ ]:
!pip install -q datasets gensim pyLDAvis wordcloud

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
import pymorphy3
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import pyLDAvis
import pyLDAvis.lda_model
from wordcloud import WordCloud
from datasets import load_dataset

nltk.download('stopwords', quiet=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Загрузка данных

In [ ]:
df = load_dataset('MonoHime/ru_sentiment_dataset', split='train').to_pandas()

text_col  = 'text'  if 'text'  in df.columns else df.columns[0]
label_col = 'label' if 'label' in df.columns else df.columns[-1]

print(df.shape)
df.head(3)

In [ ]:
print(df.dtypes)
print('\nПропуски:')
print(df.isnull().sum())

## 2. Разведывательный анализ (EDA)

### 2.1 Баланс классов

In [ ]:
counts = df[label_col].value_counts()

fig, ax = plt.subplots(figsize=(6, 3))
counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Распределение классов тональности')
ax.set_xlabel('Класс')
ax.set_ylabel('Количество')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

(counts / len(df)).round(3).rename('Доля').to_frame()

### 2.2 Длина текстов

In [ ]:
df['char_len']   = df[text_col].str.len()
df['word_count'] = df[text_col].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, col, title in zip(axes,
                           ['char_len', 'word_count'],
                           ['Длина в символах', 'Количество слов']):
    ax.hist(df[col], bins=60, edgecolor='black')
    ax.axvline(df[col].median(), color='red', linestyle='--',
               label=f'Медиана: {df[col].median():.0f}')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

df[['char_len', 'word_count']].describe().round(1)

### 2.3 Длина по классам

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, col, title in zip(axes,
                           ['char_len', 'word_count'],
                           ['Символы по классам', 'Слова по классам']):
    df.boxplot(column=col, by=label_col, ax=ax, sym='')
    ax.set_title(title)
    ax.set_xlabel('Класс')
plt.suptitle('')
plt.tight_layout()
plt.show()

### 2.4 Самые частотные слова

In [ ]:
words, freqs = zip(*Counter(' '.join(df[text_col]).lower().split()).most_common(30))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(words[::-1], freqs[::-1], edgecolor='black')
ax.set_title('Топ-30 слов (без предобработки)')
ax.set_xlabel('Частота')
plt.tight_layout()
plt.show()

## 3. Тематическое моделирование

### 3.1 Выбор алгоритма

Выбран **LDA (Latent Dirichlet Allocation)**.

LDA предполагает, что каждый документ — это смесь нескольких тем, а каждая тема — это распределение над словами. Это хорошо подходит для отзывов: один текст может одновременно говорить о цене, качестве и доставке, не принадлежа жёстко к одной теме.

Из рассматривавшихся альтернатив: NMF быстрее, но его темы менее интерпретируемы на длинных текстах; BERTopic даёт более качественные темы за счёт контекстуальных эмбеддингов, но требует GPU и тяжёлых зависимостей. LDA — разумный баланс между качеством, интерпретируемостью и простотой воспроизведения.

### 3.2 Предобработка

Для русского языка применяется лемматизация через `pymorphy3` — в отличие от стемминга, она возвращает словарную форму, а не обрезанный корень, что важно для читаемости тем.

In [ ]:
morph = pymorphy3.MorphAnalyzer()

ru_stopwords = set(stopwords.words('russian')) | {
    'это', 'так', 'вот', 'бы', 'уж', 'ну', 'же', 'ещё',
    'тоже', 'просто', 'свой', 'один', 'такой', 'очень',
    'весь', 'который', 'которые',
}

def preprocess(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^а-яёa-z\s]', ' ', text.lower())
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(
        morph.parse(w)[0].normal_form
        for w in text.split()
        if len(w) > 2 and w not in ru_stopwords
    )

df['processed'] = df[text_col].map(preprocess)
df['processed'] = df['processed'].apply(
    lambda t: ' '.join(w for w in t.split() if w not in ru_stopwords)
)

df[['processed']].head(3)

In [ ]:
words, freqs = zip(*Counter(' '.join(df['processed']).split()).most_common(30))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(words[::-1], freqs[::-1], edgecolor='black')
ax.set_title('Топ-30 слов после предобработки')
ax.set_xlabel('Частота')
plt.tight_layout()
plt.show()

### 3.3 Подбор числа тем

Перед обучением финальной модели нужно выбрать число тем. Мы обучаем LDA с разным количеством тем (от 3 до 15) и смотрим на два показателя:
- **Perplexity** — насколько хорошо модель предсказывает данные; чем ниже, тем лучше.
- **Log-likelihood** — общее правдоподобие данных; чем выше, тем лучше.

Оптимальное число тем — в точке «локтя», где улучшение показателей замедляется.

In [ ]:
vectorizer = CountVectorizer(
    max_df=0.90,
    min_df=5,
    max_features=10_000,
    ngram_range=(1, 2),
)
dtm = vectorizer.fit_transform(df['processed'])
print(f'Матрица документ–терм: {dtm.shape}')

In [ ]:
topic_range = range(3, 16)
perplexities, log_likelihoods = [], []

for n in topic_range:
    lda = LatentDirichletAllocation(
        n_components=n, random_state=42,
        max_iter=15, learning_method='online', n_jobs=-1,
    )
    lda.fit(dtm)
    perplexities.append(lda.perplexity(dtm))
    log_likelihoods.append(lda.score(dtm))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(topic_range), perplexities, marker='o')
ax1.set_title('Perplexity — чем ниже, тем лучше')
ax1.set_xlabel('Число тем')
ax1.set_ylabel('Perplexity')

ax2.plot(list(topic_range), log_likelihoods, marker='o', color='tab:orange')
ax2.set_title('Log-likelihood — чем выше, тем лучше')
ax2.set_xlabel('Число тем')
ax2.set_ylabel('Log-likelihood')

plt.tight_layout()
plt.show()

### 3.4 Обучение финальной модели

In [ ]:
N_TOPICS = 8

lda_final = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=30,
    learning_method='online',
    n_jobs=-1,
)
lda_final.fit(dtm)

doc_topics = lda_final.transform(dtm)
df['dominant_topic'] = doc_topics.argmax(axis=1)

print('Готово')

## 4. Визуализация тем

### 4.1 Топ-слова по каждой теме

In [ ]:
vocab = vectorizer.get_feature_names_out()
TOP_N = 12

fig, axes = plt.subplots(2, N_TOPICS // 2, figsize=(18, 8))

for idx, (ax, comp) in enumerate(zip(axes.flat, lda_final.components_)):
    top_idx = comp.argsort()[-TOP_N:][::-1]
    ax.barh(vocab[top_idx][::-1], comp[top_idx][::-1], edgecolor='black')
    ax.set_title(f'Тема {idx + 1}', fontsize=11)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Топ-12 слов по каждой теме', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4.2 Облака слов

In [ ]:
fig, axes = plt.subplots(2, N_TOPICS // 2, figsize=(18, 8))

for idx, (ax, comp) in enumerate(zip(axes.flat, lda_final.components_)):
    wc = WordCloud(
        width=400, height=250,
        background_color='white',
        colormap='viridis',
        max_words=60,
    ).generate_from_frequencies(dict(zip(vocab, comp)))

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Тема {idx + 1}', fontsize=11)

plt.suptitle('Облака слов по темам', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Сколько документов относится к каждой теме

In [ ]:
topic_counts = df['dominant_topic'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
topic_counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Количество документов по доминирующей теме')
ax.set_xlabel('Тема')
ax.set_ylabel('Количество')
ax.set_xticklabels([f'Тема {i+1}' for i in topic_counts.index], rotation=30)
plt.tight_layout()
plt.show()

### 4.4 Связь тем с тональностью

In [ ]:
cross = pd.crosstab(
    df['dominant_topic'],
    df[label_col],
    normalize='index',
).rename(index=lambda x: f'Тема {x+1}')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=.5, ax=ax)
ax.set_title('Доля каждого класса тональности внутри темы')
ax.set_xlabel('Класс')
ax.set_ylabel('Тема')
plt.tight_layout()
plt.show()

Тепловая карта показывает, насколько каждая тема «заряжена» тем или иным классом тональности. Если тема сильно смещена к одному классу, это говорит о содержательной связи между темой и тоном отзывов.

### 4.5 Интерактивная визуализация (pyLDAvis)

Каждый пузырь — одна тема, размер пропорционален её доле в корпусе. Расстояние между пузырями отражает схожесть тем: если они перекрываются, темы нечёткие. Ползунок λ на правой панели регулирует, насколько слово должно быть специфичным именно для этой темы (λ=1 — просто частотные слова, λ=0 — максимально уникальные).

In [ ]:
pyLDAvis.enable_notebook()

vis = pyLDAvis.lda_model.prepare(
    lda_final, dtm, vectorizer, mds='mmds', sort_topics=False
)
pyLDAvis.display(vis)

## 5. Варианты улучшения

**1. Фильтровать слова по части речи.**  
Сейчас в модель попадают все слова, включая глаголы и служебные части речи. Если оставить только существительные и прилагательные (через spaCy с моделью `ru_core_news_lg`), темы станут чище и понятнее.

**2. Подбирать число тем через когерентность.**  
Perplexity — не всегда хороший критерий: модель с минимальной perplexity не обязательно даёт осмысленные темы. Метрика `C_v coherence` из `gensim` лучше согласуется с тем, как человек оценивает качество тем, и даёт более надёжный способ выбрать оптимальное их количество.

**3. Строить отдельные модели для позитивных и негативных отзывов.**  
Сейчас модель видит все тексты вместе и смешивает темы из разных классов. Если обучить две отдельные модели, можно понять, о чём конкретно пишут довольные пользователи и о чём — недовольные.

**4. Задать начальные слова для тем (Guided LDA).**  
Если заранее известно, какие темы ожидаются (например, «цена», «доставка», «качество»), можно передать несколько ключевых слов для каждой темы. Модель будет ориентироваться на них при обучении, и темы получатся более предсказуемыми и интерпретируемыми.